# Exploration Notebook

Phase 1 scaffold notebook for ad-hoc data validation and EDA.

## Phase 2 validation

This section exercises the phase 2 science helpers against the processed parquet outputs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.loader import get_latest_snapshot, get_list_size_ts
from science import (
    cluster_practices,
    flag_anomalies,
    flag_underserved,
    forecast_list_size,
    regional_inequality,
    size_imd_correlation,
)

latest_snapshot = get_latest_snapshot()
list_size_ts = get_list_size_ts()

print('latest_snapshot shape:', latest_snapshot.shape)
print('list_size_ts shape:', list_size_ts.shape)
print('latest snapshot columns:', latest_snapshot.columns.tolist())

latest_snapshot shape: (6145, 19)
list_size_ts shape: (501614, 4)
latest snapshot columns: ['SNAPSHOT_DATE', 'CODE', 'NUMBER_OF_PATIENTS', 'DATA_SOURCE', 'PRACTICE_CODE', 'PRACTICE_NAME', 'PCN_CODE', 'PCN_NAME', 'ICB_CODE', 'ICB_NAME', 'COMM_REGION_CODE', 'COMM_REGION_NAME', 'SUPPLIER_NAME', 'CLINICAL_SYSTEM', 'POSTCODE', 'IMD_SCORE', 'IMD_DECILE', 'LATITUDE', 'LONGITUDE']


In [3]:
national_ts = (
    list_size_ts.groupby('SNAPSHOT_DATE', as_index=False)['NUMBER_OF_PATIENTS']
    .sum()
    .sort_values('SNAPSHOT_DATE')
)
forecast = forecast_list_size(national_ts.tail(24), periods=3)

assert not forecast.empty
assert {'ds', 'yhat', 'yhat_lower', 'yhat_upper'}.issubset(forecast.columns)

forecast

00:00:42 - cmdstanpy - INFO - Chain [1] start processing
00:00:48 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
0,2026-07-01,6.304087e+07,6.304087e+07,6.304087e+07
1,2026-08-01,6.292561e+07,6.292561e+07,6.292561e+07
2,2026-09-01,6.276007e+07,6.272864e+07,6.277516e+07


In [4]:
anomaly_input = list_size_ts.head(2000).copy()
anomalies = flag_anomalies(anomaly_input)

assert {'MOM_CHANGE_ABS', 'MOM_CHANGE_PCT', 'ANOMALY_TYPE', 'ANOMALY_FLAG'}.issubset(anomalies.columns)
print('anomaly rows:', len(anomalies))
print('flagged rows:', int(anomalies['ANOMALY_FLAG'].sum()))
anomalies[['CODE', 'SNAPSHOT_DATE', 'ANOMALY_TYPE', 'ANOMALY_FLAG']].dropna(subset=['ANOMALY_TYPE']).head()

anomaly rows: 2000
flagged rows: 100


,CODE,SNAPSHOT_DATE,ANOMALY_TYPE,ANOMALY_FLAG


In [5]:
cluster_input = latest_snapshot.copy()
clustered = cluster_practices(cluster_input)

assert {'CLUSTER', 'CLUSTER_LABEL', 'UMAP_X', 'UMAP_Y', 'CLUSTER_SIZE'}.issubset(clustered.columns)
print('cluster count:', clustered['CLUSTER'].nunique())
clustered[['PRACTICE_CODE', 'CLUSTER', 'CLUSTER_SIZE', 'CLUSTER_DOMINANT_SYSTEM']].head()

/workspaces/nhs-gp-analytics/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster count: 5


,PRACTICE_CODE,CLUSTER,CLUSTER_SIZE,CLUSTER_DOMINANT_SYSTEM
0,A81058,0,1299,EMIS Web
1,A81611,0,1299,EMIS Web
2,A82003,0,1299,EMIS Web
3,A82005,0,1299,EMIS Web
4,A82021,0,1299,EMIS Web


In [6]:
deprivation_input = latest_snapshot.copy()
underserved = flag_underserved(deprivation_input)
inequality = regional_inequality(deprivation_input)
correlation = size_imd_correlation(deprivation_input)

assert {'UNDER_SERVED', 'NATIONAL_MEDIAN_PATIENTS', 'DEPRIVED_AREA', 'SMALL_PRACTICE'}.issubset(underserved.columns)
assert {'GINI_COEFFICIENT', 'PRACTICE_COUNT', 'MEAN_PATIENTS', 'MEDIAN_PATIENTS'}.issubset(inequality.columns)
assert {'PEARSON_R', 'P_VALUE', 'N'}.issubset(correlation.columns)

print('underserved rows:', len(underserved))
print('inequality rows:', len(inequality))
print('correlation rows:', len(correlation))
underserved[underserved['UNDER_SERVED']].head()

underserved rows: 6145
inequality rows: 70
correlation rows: 7


,SNAPSHOT_DATE,CODE,NUMBER_OF_PATIENTS,DATA_SOURCE,PRACTICE_CODE,PRACTICE_NAME,PCN_CODE,PCN_NAME,ICB_CODE,ICB_NAME,...,CLINICAL_SYSTEM,POSTCODE,IMD_SCORE,IMD_DECILE,LATITUDE,LONGITUDE,NATIONAL_MEDIAN_PATIENTS,DEPRIVED_AREA,SMALL_PRACTICE,UNDER_SERVED
0,2026-06-01,A81001,3753,PDS,A81001,THE DENSHAM SURGERY,U89141,STOCKTON PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS181HU,77.084,1.0,54.561637,-1.318999,8814.0,True,True,True
3,2026-06-01,A81005,7540,PDS,A81005,SPRINGWOOD SURGERY,U07842,EAST CLEVELAND PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS147DJ,30.334,3.0,54.532610,-1.055459,8814.0,True,True,True
6,2026-06-01,A81009,7835,PDS,A81009,VILLAGE MEDICAL CENTRE,U85008,HOLGATE PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS56HF,57.687,1.0,54.562285,-1.241897,8814.0,True,True,True
8,2026-06-01,A81012,5617,PDS,A81012,WESTBOURNE MEDICAL CENTRE,U02671,GREATER MIDDLESBROUGH PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS36AL,65.896,1.0,54.571738,-1.216246,8814.0,True,True,True
10,2026-06-01,A81014,4097,PDS,A81014,QUEENSTREE PRACTICE,U94460,BILLINGHAM & NORTON PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS232LA,42.007,2.0,54.608303,-1.294813,8814.0,True,True,True
